# ChargeGrid Intelligence Bot — Sprint 2
### Chatbot operacional para gerenciamento de eletropostos GoodWe — EV Challenge 2026 | FIAP

---

**Integrantes:**
| Nome | RM |
|------|-----|
| Caio César Portela França | 573127 |
| Davi Teodoro Novais | 571022 |
| Gustavo Curis de Francisco | 569704 |
| Lourenco Borges da Silva | 569515 |
| Tiago Pimentel Muniz | 574148 |

---

**Arquitetura implementada:**
- `GPT-4o-mini` via OpenAI API
- System prompt com contexto operacional GoodWe
- Memória de histórico de mensagens por sessão
- **RAG (Retrieval-Augmented Generation)** com base de conhecimento vetorizada via FAISS
- API Key via Google Colab Secrets (nunca exposta no código)

In [ ]:
!pip install openai faiss-cpu numpy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 73.2 MB/s eta 0:00:00


In [ ]:
import json
import numpy as np
import faiss
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

print("✅ Cliente OpenAI configurado com sucesso.")

✅ Cliente OpenAI configurado com sucesso.


In [ ]:
with open('knowledge_base.json', 'r', encoding='utf-8') as f:
    knowledge_base = json.load(f)

print(f"📚 Base de conhecimento carregada: {len(knowledge_base)} documentos")


def get_embedding(text: str) -> list[float]:
    """Converte texto em vetor numérico usando o modelo de embeddings da OpenAI."""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

print("⏳ Gerando embeddings da base de conhecimento...")


doc_texts = [f"{doc['titulo']}\n{doc['conteudo']}" for doc in knowledge_base]
doc_embeddings = [get_embedding(text) for text in doc_texts]

print(f"✅ {len(doc_embeddings)} embeddings gerados.")


embedding_dim = len(doc_embeddings[0])
embeddings_matrix = np.array(doc_embeddings, dtype=np.float32)


faiss.normalize_L2(embeddings_matrix)

index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings_matrix)

print(f"✅ Índice FAISS construído — {index.ntotal} vetores indexados.")

📚 Base de conhecimento carregada: 25 documentos
⏳ Gerando embeddings da base de conhecimento...
✅ 25 embeddings gerados.
✅ Índice FAISS construído — 25 vetores indexados.


In [ ]:
SYSTEM_PROMPT_BASE = """Você é o ChargeGrid Intelligence Bot, assistente operacional especializado \
em eletropostos comerciais da GoodWe para o EV Challenge 2026 | FIAP.

Seu interlocutor é sempre um OPERADOR COMERCIAL — responsável pela gestão diária do eletroposto. \
Trate-o como profissional competente que precisa de respostas objetivas e acionáveis.

## Seu papel
Você auxilia o operador a:
- Entender o STATUS dos conectores e sessões de recarga em andamento
- Interpretar dados de POTÊNCIA e orquestração de carga (balanceamento, throttling)
- Consultar HISTÓRICO de ciclos de recarga e consumo energético
- Compreender COBRANÇAS e faturamento por sessão ou período
- Interpretar ALERTAS e códigos de erro dos equipamentos GoodWe
- Seguir PROCEDIMENTOS operacionais padrão

## Regras de comportamento
1. SEJA DIRETO — respostas longas sem ação clara são inúteis no campo.
2. QUANDO NÃO SOUBER, diga isso e indique o próximo passo.
3. NUNCA invente dados técnicos ou números que não constem no contexto fornecido.
4. USE LINGUAGEM TÉCNICA ADEQUADA: kW, kWh, OCPP, Tipo 2, CHAdeMO.
5. PRIORIZE SEGURANÇA: risco elétrico → oriente desligar e acionar suporte imediatamente.
6. HISTÓRICO: considere mensagens anteriores para manter coerência.
7. IDIOMA: responda sempre em português do Brasil.

## O que você NÃO faz
- Não emite comandos diretos aos equipamentos
- Não realiza manutenção remota
- Não discute assuntos fora do escopo do eletroposto GoodWe"""


def retrieve_context(query: str, top_k: int = 3) -> str:
    """
    RAG — Recuperação: busca os top_k documentos mais relevantes
    da base de conhecimento para a pergunta do usuário.
    """
    query_embedding = np.array([get_embedding(query)], dtype=np.float32)
    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, top_k)

    retrieved_docs = []
    for score, idx in zip(scores[0], indices[0]):
        if score > 0.3:
            doc = knowledge_base[idx]
            retrieved_docs.append(
                f"[{doc['categoria'].upper()}] {doc['titulo']}:\n{doc['conteudo']}"
            )

    if not retrieved_docs:
        return ""

    return "\n\n".join(retrieved_docs)


def build_system_prompt(context: str) -> str:
    """
    RAG — Geração aumentada: injeta o contexto recuperado no system prompt.
    """
    if not context:
        return SYSTEM_PROMPT_BASE

    return SYSTEM_PROMPT_BASE + f"""

## Contexto técnico relevante para esta pergunta
(Informações recuperadas da base de conhecimento GoodWe — use-as para embasar sua resposta)

{context}"""


def chat(user_message: str, history: list) -> tuple[str, list]:
    """
    Envia mensagem ao modelo com:
    - Contexto RAG recuperado dinamicamente
    - Histórico completo da conversa (memória)
    Retorna a resposta e o histórico atualizado.
    """

    context = retrieve_context(user_message)


    system_prompt = build_system_prompt(context)


    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(history)
    messages.append({"role": "user", "content": user_message})


    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.3,
        max_tokens=600
    )

    assistant_reply = response.choices[0].message.content


    history.append({"role": "user", "content": user_message})
    history.append({"role": "assistant", "content": assistant_reply})

    return assistant_reply, history


print("✅ Funções de RAG e chat configuradas.")

✅ Funções de RAG e chat configuradas.


In [ ]:
casos_de_teste = [
    {
        "id": "TC01",
        "descricao": "Consulta de status de conector e indicadores LED",
        "pergunta": "O LED do conector 2 está amarelo piscando. O que significa e o que devo fazer?"
    },
    {
        "id": "TC02",
        "descricao": "Interpretação de erro de equipamento",
        "pergunta": "Apareceu o erro E02 no display do HCA-G2. O carregamento parou. O que aconteceu?"
    },
    {
        "id": "TC03",
        "descricao": "Consulta de histórico e faturamento por período",
        "pergunta": "Como faço para ver quantas sessões de recarga ocorreram hoje no período da tarde e quanto foi faturado?"
    },
    {
        "id": "TC04",
        "descricao": "Dúvida sobre balanceamento de carga com dois veículos",
        "pergunta": "Dois veículos estão carregando ao mesmo tempo no HCA-G2. A potência é dividida entre eles? Como funciona esse balanceamento?"
    },
    {
        "id": "TC05",
        "descricao": "Procedimento de emergência por risco elétrico",
        "pergunta": "Estou sentindo cheiro de queimado perto do HCA-G2 e vi uma faísca. O que faço agora?"
    }
]

resultados_teste = []

print("=" * 70)
print(" EXECUÇÃO DOS CASOS DE TESTE — ChargeGrid Intelligence Bot")
print("=" * 70)

for caso in casos_de_teste:

    historico_teste = []

    print(f"\n{'─' * 70}")
    print(f"🧪 {caso['id']} — {caso['descricao']}")
    print(f"{'─' * 70}")
    print(f"📨 PERGUNTA:\n{caso['pergunta']}")

    resposta, historico_teste = chat(caso['pergunta'], historico_teste)

    print(f"\n🤖 RESPOSTA:\n{resposta}")
    print(f"{'─' * 70}")

    resultados_teste.append({
        "id": caso["id"],
        "descricao": caso["descricao"],
        "pergunta": caso["pergunta"],
        "resposta": resposta
    })

print("\n✅ Todos os casos de teste executados.")

 EXECUÇÃO DOS CASOS DE TESTE — ChargeGrid Intelligence Bot

──────────────────────────────────────────────────────────────────────
🧪 TC01 — Consulta de status de conector e indicadores LED
──────────────────────────────────────────────────────────────────────
📨 PERGUNTA:
O LED do conector 2 está amarelo piscando. O que significa e o que devo fazer?

🤖 RESPOSTA:
O LED amarelo piscando no conector 2 indica que o throttling está ativo ou que há um aviso de manutenção. 

Ações recomendadas:
1. Verifique se há alguma limitação de potência configurada no sistema que possa estar causando o throttling.
2. Inspecione o conector e o cabo para garantir que não haja sujeira ou danos visíveis.
3. Consulte o display LCD para verificar se há mensagens adicionais ou informações sobre o status da sessão de recarga.

Se o problema persistir ou se você suspeitar de uma falha, entre em contato com o suporte técnico da GoodWe para assistência.
───────────────────────────────────────────────────────────────

In [ ]:
print("=" * 60)
print(" ChargeGrid Intelligence Bot — GoodWe EV Challenge 2026")
print("=" * 60)
print("Assistente operacional para eletropostos GoodWe.")
print("Digite 'sair' para encerrar a sessão.")
print("-" * 60)

historico = []

while True:
    try:
        pergunta = input("\n👤 Operador: ").strip()
    except EOFError:
        break

    if not pergunta:
        continue

    if pergunta.lower() == "sair":
        print("\n🔌 Sessão encerrada. Até logo!")
        break

    resposta, historico = chat(pergunta, historico)
    print(f"\n🤖 ChargeGrid Bot:\n{resposta}")

 ChargeGrid Intelligence Bot — GoodWe EV Challenge 2026
Assistente operacional para eletropostos GoodWe.
Digite 'sair' para encerrar a sessão.
------------------------------------------------------------

👤 Operador: Qual o carregador de carros elétricos da GoodWe?

🤖 ChargeGrid Bot:
O carregador de carros elétricos da GoodWe é o HCA-G2. Ele é um carregador AC comercial, projetado para estacionamentos e frotas, e opera em corrente alternada, com potência configurável de 7,4 kW a 22 kW por conector. O HCA-G2 possui dois conectores Tipo 2 e é compatível com a comunicação OCPP 1.6 JSON.

👤 Operador: sair

🔌 Sessão encerrada. Até logo!
